# PlantCLEF 2015 Leaf S-CNN Training

Clean Colab workflow for the current project state. Use the Google Drive leaf-only archive first, then smoke-train, evaluate `S-CNN (A)`, and only then run longer training.

## 1. Runtime Check

Select `Runtime -> Change runtime type -> GPU` before running training cells.

In [27]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CUDA: True
Device: Tesla T4


## 2. Clone Or Update Project

In [ ]:
%cd /content
!rm -rf diploma
!git clone -b robodanill/main https://github.com/robodanill/diploma.git
%cd /content/diploma
!pip install -e '.[ml]'


In [28]:
%cd /content/diploma
!git pull
!pip install -e '.[ml]'

/content/diploma
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 14 (delta 10), reused 14 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 4.23 KiB | 867.00 KiB/s, done.
From https://github.com/robodanill/diploma
   301bcb7..2c3f59c  robodanill/main -> origin/robodanill/main
Updating 301bcb7..2c3f59c
Fast-forward
 PROJECT_NOTES.md                         |   4 +-
 configs/leaf_training.yaml               |   2 +
 configs/smoke_training.yaml              |   2 +
 configs/training.yaml                    |   3 +-
 notebooks/plantclef_colab_training.ipynb | 280 +++++++++----------------------
 src/plant_classifier/training/cli.py     |  81 +++++----
 src/plant_classifier/training/loop.py    |  90 ++++++++++
 7 files changed, 232 insertions(+), 230 deletions(-)
Obtaining file:///content/diploma
  Installing build dependencies ... done
  Checking if build backend supports bui

## 3. Mount Google Drive, Unpack Leaf Dataset, And Create Split

Expected archive path: `/content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz`. This creates `metadata_split.csv` with train/val/test labels.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [29]:
%cd /content/diploma
!rm -rf data/plantclef2015
!mkdir -p data/plantclef2015
!tar -xzf /content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz -C data/plantclef2015
!cp data/plantclef2015/leaf/metadata.csv data/plantclef2015/metadata.csv
!plant-classifier-split-metadata \
  --metadata data/plantclef2015/metadata.csv \
  --dataset-root data/plantclef2015/leaf \
  --output data/plantclef2015/metadata_split.csv \
  --train-ratio 0.70 \
  --val-ratio 0.15 \
  --test-ratio 0.15
!wc -l data/plantclef2015/metadata_split.csv
!python - <<'PY2'
import csv
from collections import Counter
rows = list(csv.DictReader(open('data/plantclef2015/metadata_split.csv')))
print(Counter(row['split'] for row in rows))
PY2


/content/diploma
13368 data/plantclef2015/metadata.csv
13367


## 4. Smoke Train `S-CNN (A)` Genus

This is only a pipeline check on a small subset. It also runs genus retrieval evaluation after the epoch.


In [30]:
!plant-classifier-train   --config configs/smoke_training.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_vgg16.pt


using subset: 120 images from 20 species
epoch=1 loss=0.7291 best_loss=0.7291 time=7.0s best


## 5. Smoke Evaluate `S-CNN (A)` On Validation Split


In [31]:
!plant-classifier-eval-genus   --config configs/smoke_training.yaml   --checkpoint checkpoints/smoke_scnn_genus_vgg16.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


references=48 queries=48 top_k=5
top5_genus_accuracy=0.271 (13/48)
query genus distribution: {'Abies': 2, 'Acacia': 2, 'Acanthus': 2, 'Acer': 2, 'Achillea': 2, 'Aconitum': 2, 'Adenostyles': 2, 'Aegopodium': 2, 'Aesculus': 2, 'Agave': 2}
reference genus distribution: {'Abies': 2, 'Acacia': 2, 'Acanthus': 2, 'Acer': 2, 'Achillea': 2, 'Aconitum': 2, 'Adenostyles': 2, 'Aegopodium': 2, 'Aesculus': 2, 'Agave': 2}


## 6. Full Leaf Train `S-CNN (A)` Genus

Use this after the smoke path works. Training re-samples pairs every epoch, evaluates genus top-k retrieval after each epoch, and saves `scnn_genus_vgg16_best.pt` by best retrieval accuracy.


In [32]:
!plant-classifier-train   --config configs/leaf_training.yaml   --stage genus   --output checkpoints/scnn_genus_vgg16.pt


epoch=1 loss=0.7129 best_loss=0.7129 time=30.2s best
epoch=2 loss=0.6866 best_loss=0.6866 time=32.8s best
epoch=3 loss=0.6745 best_loss=0.6745 time=35.8s best
epoch=4 loss=0.6799 best_loss=0.6745 time=34.0s
epoch=5 loss=0.6666 best_loss=0.6666 time=34.2s best
Traceback (most recent call last):
  File "/usr/local/bin/plant-classifier-train", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/diploma/src/plant_classifier/training/cli.py", line 48, in main
    train_siamese_with_dynamic_pairs(
  File "/content/diploma/src/plant_classifier/training/loop.py", line 133, in train_siamese_with_dynamic_pairs
    running_loss += loss.item()
                    ^^^^^^^^^^^
KeyboardInterrupt


## 7. Evaluate Full `S-CNN (A)` On Validation Split


In [33]:
!plant-classifier-eval-genus   --config configs/leaf_training.yaml   --checkpoint checkpoints/scnn_genus_vgg16_best.pt   --max-species 120   --references-per-genus 2   --queries-per-genus 2   --top-k 5


references=144 queries=144 top_k=5
top5_genus_accuracy=0.285 (41/144)
query genus distribution: {'Abies': 2, 'Acacia': 2, 'Acanthus': 2, 'Acer': 2, 'Achillea': 2, 'Aconitum': 2, 'Adenostyles': 2, 'Aegopodium': 2, 'Aesculus': 2, 'Agave': 2}
reference genus distribution: {'Abies': 2, 'Acacia': 2, 'Acanthus': 2, 'Acer': 2, 'Achillea': 2, 'Aconitum': 2, 'Adenostyles': 2, 'Aegopodium': 2, 'Aesculus': 2, 'Agave': 2}


## 8. Sync Checkpoints To Google Drive

Old Drive checkpoints are removed unless their name contains `_best`. Local `*_best` files are copied to Drive as regular checkpoint names, so Drive `*_best` files can mean best-across-all-runs.

In [ ]:
!python scripts/sync_checkpoints_to_drive.py   --source checkpoints   --dest /content/drive/MyDrive/diploma_checkpoints   --keep-token _best
!ls -lh /content/drive/MyDrive/diploma_checkpoints


## 9. Train `S-CNN (B)` Species

Run this only after `S-CNN (A)` has a reasonable genus retrieval result.

In [ ]:
!plant-classifier-train   --config configs/leaf_training.yaml   --stage species   --output checkpoints/scnn_species_vgg16.pt


## 10. Build Reference Index For Desktop Inference

In [ ]:
!plant-classifier-build-index   --config configs/leaf_training.yaml   --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt   --species-checkpoint checkpoints/scnn_species_vgg16_best.pt   --output checkpoints/reference_index.pt
!ls -lh checkpoints


## 11. Final Sync To Google Drive

In [ ]:
!python scripts/sync_checkpoints_to_drive.py   --source checkpoints   --dest /content/drive/MyDrive/diploma_checkpoints   --keep-token _best
!ls -lh /content/drive/MyDrive/diploma_checkpoints
